# Tiny VGG Multi-Model Training Notebook

This notebook trains **three different CNN architectures with different layer orders** for the CNN Explainer project and exports all of them in a frontend-friendly format.

The default set covers three different depths:

- **Compact 7-layer**: a Conv → Pool → ReLU style order
- **Balanced 12-layer**: a mixed Conv / Pool / ReLU order
- **Deep 17-layer**: a deeper mixed-order CNN with three pooling stages

The notebook will:

1. mount Google Drive and prepare the dataset,
2. define several different architectures,
3. train and evaluate each model,
4. export each one to TensorFlow.js,
5. write a minimal `model-index.json` manifest for the website selector.

> Task 6.1: the examples below now vary both **layer order** and the **number of convolution kernels**. The number of kernels is the number of filters/nodes shown in each convolutional feature layer, not the spatial kernel size.

> Important: TensorFlow.js `.bin` files are raw weight shards. They do not contain layer names, layer order, tensor names, tensor shapes, or model topology. The website can derive architecture from `model.json`; it cannot derive architecture from `.bin` alone.


In [ ]:
# Colab / environment setup
from pathlib import Path
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from glob import glob
from time import time

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Keep TensorFlow, TensorFlow.js converter, and TensorFlow Decision Forests ABI-compatible.
    # Run this before TensorFlow is imported. If TensorFlow was already imported, restart the runtime first.
    subprocess.run([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        'tensorflow==2.19.0',
        'tensorflowjs==4.22.0',
        'tensorflow-decision-forests==1.12.0',
    ], check=True)

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Activation, MaxPool2D, AveragePooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print('TensorFlow:', tf.__version__)

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/CNN')
    DATA_ZIP = DRIVE_PROJECT_DIR / 'data.zip'
    WORKDIR = Path('/content/tiny-vgg')
else:
    WORKDIR = Path.cwd()
    DRIVE_PROJECT_DIR = WORKDIR
    DATA_ZIP = WORKDIR / 'data.zip'

WORKDIR.mkdir(parents=True, exist_ok=True)

print('Working directory:', WORKDIR)
print('Project directory:', DRIVE_PROJECT_DIR)
print('Data zip:', DATA_ZIP)
print('Data zip exists:', DATA_ZIP.exists())

TensorFlow: 2.19.0


ValueError: mount failed

## Data Setup

This notebook expects the same dataset layout used by `tiny-vgg.py`.

Preferred layout after extraction:

```text
tiny-vgg/
  data/
    class_dict_10.json
    val_class_dict_10.json
    class_10_train/
    class_10_val/
```

If only `data.zip` exists, the next cell extracts it.


In [ ]:
DATA_DIR = WORKDIR / 'data'

if not DATA_ZIP.exists():
    raise FileNotFoundError(f'Could not find {DATA_ZIP}')

if not DATA_DIR.exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        zf.extractall(WORKDIR)
    print('Extracted:', DATA_ZIP)

print('DATA_DIR exists:', DATA_DIR.exists())
if DATA_DIR.exists():
    print('Data directory:', DATA_DIR)


Extracted: /content/drive/MyDrive/CNN/data.zip
DATA_DIR exists: True
Data directory: /content/tiny-vgg/data


In [ ]:
# Shared hyperparameters
WIDTH = 64
HEIGHT = 64
NUM_CLASS = 10
FILTERS = 10
EPOCHS = 25
PATIENCE = 8
LR = 1e-3
BATCH_SIZE = 64
SHUFFLE_BUFFER = 1000
# HIDDEN_DENSE_UNITS = [16, 8]
HIDDEN_DENSE_UNITS = [64, 32]
# HIDDEN_DENSE_UNITS = [128, 64]

MODEL_ROOT = WORKDIR / 'trained_models'
TFJS_ROOT = WORKDIR / 'frontend_models'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
TFJS_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
# Dataset metadata
class_dict_path = DATA_DIR / 'class_dict_10.json'
val_class_dict_path = DATA_DIR / 'val_class_dict_10.json'

assert class_dict_path.exists(), f'Missing {class_dict_path}'
assert val_class_dict_path.exists(), f'Missing {val_class_dict_path}'

with open(class_dict_path, 'r') as f:
    tiny_class_dict = json.load(f)
with open(val_class_dict_path, 'r') as f:
    tiny_val_class_dict = json.load(f)

train_pattern = str(DATA_DIR / 'class_10_train' / '*' / 'images' / '*.JPEG')
val_pattern = str(DATA_DIR / 'class_10_val' / 'val_images' / '*.JPEG')
test_pattern = str(DATA_DIR / 'class_10_val' / 'test_images' / '*.JPEG')

print('Train images:', len(glob(train_pattern)))
print('Val images:', len(glob(val_pattern)))
print('Test images:', len(glob(test_pattern)))


Train images: 5000
Val images: 250
Test images: 250


In [ ]:
def process_path_train(path):
    path = path.numpy()
    image_name = os.path.basename(path.decode('ascii'))
    label_name = re.sub(r'(.+)_\d+\.JPEG', r'\1', image_name)
    label_index = tiny_class_dict[label_name]['index']

    label = tf.one_hot(indices=[label_index], depth=NUM_CLASS)
    label = tf.reshape(label, [NUM_CLASS])

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [WIDTH, HEIGHT])
    return img, label


def process_path_test(path):
    path = path.numpy()
    image_name = os.path.basename(path.decode('ascii'))
    label_index = tiny_val_class_dict[image_name]['index']

    label = tf.one_hot(indices=[label_index], depth=NUM_CLASS)
    label = tf.reshape(label, [NUM_CLASS])

    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [WIDTH, HEIGHT])
    return img, label


def build_dataset(file_pattern, mode='train'):
    ds = tf.data.Dataset.list_files(file_pattern, shuffle=(mode == 'train'))

    if mode == 'train':
        mapper = lambda path: tf.py_function(process_path_train, [path], [tf.float32, tf.float32])
    else:
        mapper = lambda path: tf.py_function(process_path_test, [path], [tf.float32, tf.float32])

    ds = ds.map(mapper, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y: (tf.ensure_shape(x, [WIDTH, HEIGHT, 3]), tf.ensure_shape(y, [NUM_CLASS])))

    if mode == 'train':
        ds = ds.shuffle(SHUFFLE_BUFFER)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_dataset = build_dataset(train_pattern, mode='train')
val_dataset = build_dataset(val_pattern, mode='val')
test_dataset = build_dataset(test_pattern, mode='test')


## Model Definitions

Each entry below defines a different network depth. They all keep the same input size, class count, and number of channels per feature layer so the website can still render them consistently.


In [ ]:
# Task 3 at lines 446-467: define the three trained demonstration architectures
# that the website will later switch between for 7-layer, 12-layer, and 17-layer comparisons.
MODEL_SPECS = [
    {
        'id': 'compact-7',
        'label': 'Compact 9-layer',
        'filters': 6,
        'hidden_dense_units': HIDDEN_DENSE_UNITS,
        'layer_plan': [
            'conv', 'relu',
            'conv', 'sigmoid',
            'avg_pool',
        ],
    },
    {
        'id': 'balanced-12',
        'label': 'Balanced 14-layer',
        'filters': 10,
        'hidden_dense_units': HIDDEN_DENSE_UNITS,
        'layer_plan': [
            'conv', 'relu',
            'conv', 'relu',
            'avg_pool',

            'conv', 'relu',
            'conv', 'relu',
            'max_pool',
        ],
    },
    {
        'id': 'deep-17',
        'label': 'Deep 19-layer',
        'filters': 14,
        'hidden_dense_units': HIDDEN_DENSE_UNITS,
        'layer_plan': [
            'conv', 'relu',
            'conv', 'relu',
            'avg_pool',

            'conv', 'relu',
            'conv', 'relu',
            'max_pool',

            'conv', 'relu',
            'conv', 'relu',
            'avg_pool',
        ],
    },
]

def summarize_plan(layer_plan, hidden_dense_units):
    conv_count = layer_plan.count('conv')
    relu_count = layer_plan.count('relu')
    sigmoid_count = layer_plan.count('sigmoid')
    max_pool_count = layer_plan.count('max_pool')
    avg_pool_count = layer_plan.count('avg_pool')
    total_layers = len(layer_plan) + len(hidden_dense_units) + 2  # flatten + output
    summary = f'{conv_count} conv, {relu_count} relu, {sigmoid_count} sigmoid, {max_pool_count} max-pool, {avg_pool_count} avg-pool, {len(hidden_dense_units)} hidden dense'
    return total_layers, summary

for spec in MODEL_SPECS:
    spec['total_layers'], spec['summary'] = summarize_plan(
        spec['layer_plan'],
        spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS),
    )

pd.DataFrame([{
    'id': spec['id'],
    'label': spec['label'],
    'total_layers': spec['total_layers'],
    'conv_kernels': spec['filters'],
    'hidden_dense_units': ' -> '.join(map(str, spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS))),
    'summary': spec['summary'],
    'layer_plan': ' -> '.join(spec['layer_plan']) + ' -> flatten -> ' + ' -> '.join([
        f'dense_{idx + 1}' for idx in range(len(spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS)))
    ]) + ' -> output',
} for spec in MODEL_SPECS])


,id,label,total_layers,conv_kernels,hidden_dense_units,summary,layer_plan
0,compact-7,Compact 9-layer,9,6,16 -> 8,"2 conv, 1 relu, 1 sigmoid, 0 max-pool, 1 avg-p...",conv -> avg_pool -> sigmoid -> conv -> relu ->...
1,balanced-12,Balanced 14-layer,14,10,16 -> 8,"4 conv, 2 relu, 2 sigmoid, 1 max-pool, 1 avg-p...",conv -> avg_pool -> sigmoid -> conv -> relu ->...
2,deep-17,Deep 19-layer,19,14,16 -> 8,"6 conv, 3 relu, 3 sigmoid, 1 max-pool, 2 avg-p...",conv -> avg_pool -> sigmoid -> conv -> relu ->...


In [ ]:
SUPPORTED_LAYERS = {'conv', 'relu', 'sigmoid', 'max_pool', 'avg_pool'}

# Task 3 at lines 502-531: build a CNN directly from a layer plan so changing
# the order of conv / relu / max-pool produces a different trained demo model.
def build_configurable_cnn(
    layer_plan,
    filters=FILTERS,
    num_class=NUM_CLASS,
    hidden_dense_units=HIDDEN_DENSE_UNITS,
):
    unknown = [layer for layer in layer_plan if layer not in SUPPORTED_LAYERS]
    if unknown:
        raise ValueError(f'Unsupported layer types: {unknown}')

    layers = []
    block_index = 1
    conv_in_block = 0
    relu_in_block = 0
    sigmoid_in_block = 0

    for idx, layer_type in enumerate(layer_plan):
        first_layer_kwargs = {'input_shape': (WIDTH, HEIGHT, 3)} if idx == 0 else {}

        if layer_type == 'conv':
            conv_in_block += 1
            layers.append(
                Conv2D(filters, (3, 3), name=f'conv_{block_index}_{conv_in_block}', **first_layer_kwargs)
            )
        elif layer_type == 'relu':
            relu_in_block += 1
            layers.append(
                Activation('relu', name=f'relu_{block_index}_{relu_in_block}', **first_layer_kwargs)
            )
        elif layer_type == 'sigmoid':
            sigmoid_in_block += 1
            layers.append(
                Activation('sigmoid', name=f'sigmoid_{block_index}_{sigmoid_in_block}', **first_layer_kwargs)
            )
        elif layer_type == 'max_pool':
            layers.append(
                MaxPool2D((2, 2), name=f'max_pool_{block_index}', **first_layer_kwargs)
            )
            block_index += 1
            conv_in_block = 0
            relu_in_block = 0
            sigmoid_in_block = 0
        elif layer_type == 'avg_pool':
            layers.append(
                AveragePooling2D((2, 2), name=f'avg_pool_{block_index}', **first_layer_kwargs)
            )
            block_index += 1
            conv_in_block = 0
            relu_in_block = 0
            sigmoid_in_block = 0

    layers.append(Flatten(name='flatten'))

    for dense_index, units in enumerate(hidden_dense_units, start=1):
        layers.append(Dense(units, activation='relu', name=f'dense_{dense_index}'))

    layers.append(Dense(num_class, activation='softmax', name='output'))
    return Sequential(layers)


## Train All Architectures

This cell loops over the three architecture specs, trains each model, saves the best checkpoint, and records evaluation results.


In [ ]:
trained_models = {}
training_runs = []

# Task 3 at lines 2076-2141: train each planned architecture so the frontend
# can switch among multiple real networks instead of one fixed demo model.
for spec in MODEL_SPECS:
    tf.keras.backend.clear_session()

    model_dir = MODEL_ROOT / spec['id']
    model_dir.mkdir(parents=True, exist_ok=True)
    best_model_h5 = model_dir / 'best.h5'
    final_model_h5 = model_dir / 'final.h5'

    # Task 6.1: each model can use a different number of convolution kernels.
    model = build_configurable_cnn(
        spec['layer_plan'],
        filters=spec.get('filters', FILTERS),
        hidden_dense_units=spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS),
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss='categorical_crossentropy',
        metrics=['categorical_accuracy'],
    )

    print('=' * 100)
    print(spec['label'])
    print('Layer plan:', spec['layer_plan'] + ['flatten'] + [
        f'dense_{idx + 1}' for idx in range(len(spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS)))
    ] + ['output'])
    print('Convolution kernels per conv layer:', spec.get('filters', FILTERS))
    print('Hidden dense units:', spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS))
    model.summary()

    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=PATIENCE,
            restore_best_weights=True,
            verbose=1,
        ),
        ModelCheckpoint(
            filepath=str(best_model_h5),
            monitor='val_loss',
            save_best_only=True,
            verbose=1,
        ),
    ]

    start_time = time()
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=1,
    )
    elapsed_minutes = (time() - start_time) / 60

    if best_model_h5.exists():
        model = tf.keras.models.load_model(best_model_h5)

    model.save(final_model_h5)
    test_loss, test_acc = model.evaluate(test_dataset, verbose=1)

    trained_models[spec['id']] = model
    training_runs.append({
        'id': spec['id'],
        'label': spec['label'],
        'total_layers': spec['total_layers'],
        'summary': spec['summary'],
        'conv_kernels': spec.get('filters', FILTERS),
        'hidden_dense_units': spec.get('hidden_dense_units', HIDDEN_DENSE_UNITS),
        'train_minutes': round(elapsed_minutes, 2),
        'test_loss': float(test_loss),
        'test_accuracy': float(test_acc),
        'best_model_h5': str(best_model_h5),
        'final_model_h5': str(final_model_h5),
        'layer_names': [layer.name for layer in model.layers],
    })

summary_df = pd.DataFrame(training_runs).sort_values('total_layers')
summary_df


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Compact 9-layer
Layer plan: ['conv', 'avg_pool', 'sigmoid', 'conv', 'relu', 'flatten', 'dense_1', 'dense_2', 'output']
Convolution kernels per conv layer: 6
Hidden dense units: [16, 8]


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_1_1 (Conv2D)               │ (None, 62, 62, 6)      │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ avg_pool_1 (AveragePooling2D)   │ (None, 31, 31, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sigmoid_2_1 (Activation)        │ (None, 31, 31, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_2_1 (Conv2D)               │ (None, 29, 29, 6)      │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ relu_2_1 (Activation)           │ (None, 29, 29, 6)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 5046)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │        80,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │            90 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,476 (318.27 KB)

...


Epoch 1: finished saving model to /content/tiny-vgg/trained_models/deep-17/best.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 178ms/step - categorical_accuracy: 0.0968 - loss: 2.3063 - val_categorical_accuracy: 0.1200 - val_loss: 2.3023
Epoch 2/25
78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - categorical_accuracy: 0.1014 - loss: 2.3028
Epoch 2: val_loss did not improve from 2.30230
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 123ms/step - categorical_accuracy: 0.1012 - loss: 2.3029 - val_categorical_accuracy: 0.1200 - val_loss: 2.3025
Epoch 3/25
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - categorical_accuracy: 0.0974 - loss: 2.3035
Epoch 3: val_loss improved from 2.30230 to 2.30224, saving model to /content/tiny-vgg/trained_models/deep-17/best.h5



Epoch 3: finished saving model to /content/tiny-vgg/trained_models/deep-17/best.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 130ms/step - categorical_accuracy: 0.0966 - loss: 2.3036 - val_categorical_accuracy: 0.1080 - val_loss: 2.3022
Epoch 4/25
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - categorical_accuracy: 0.1029 - loss: 2.3031
Epoch 4: val_loss did not improve from 2.30224
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 130ms/step - categorical_accuracy: 0.1020 - loss: 2.3029 - val_categorical_accuracy: 0.0800 - val_loss: 2.3032
Epoch 5/25
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - categorical_accuracy: 0.0950 - loss: 2.3029
Epoch 5: val_loss did not improve from 2.30224
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 131ms/step - categorical_accuracy: 0.0984 - loss: 2.3028 - val_categorical_accuracy: 0.1000 - val_loss: 2.3030
Epoch 6/25
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - categorical_accuracy: 0.0883 - loss: 2.3032
Epoch 6: val_loss did not improve from 2.30224
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 130ms/step - categorical


Epoch 7: finished saving model to /content/tiny-vgg/trained_models/deep-17/best.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 21s 134ms/step - categorical_accuracy: 0.0958 - loss: 2.3031 - val_categorical_accuracy: 0.1080 - val_loss: 2.3007
Epoch 8/25
76/79 ━━━━━━━━━━━━━━━━━━━━ 0s 120ms/step - categorical_accuracy: 0.1090 - loss: 2.3026
Epoch 8: val_loss did not improve from 2.30068
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 125ms/step - categorical_accuracy: 0.1024 - loss: 2.3030 - val_categorical_accuracy: 0.0840 - val_loss: 2.3027
Epoch 9/25
76/79 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - categorical_accuracy: 0.1081 - loss: 2.3029
Epoch 9: val_loss did not improve from 2.30068
79/79 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - categorical_accuracy: 0.0994 - loss: 2.3029 - val_categorical_accuracy: 0.1000 - val_loss: 2.3031
Epoch 10/25
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - categorical_accuracy: 0.0998 - loss: 2.3029
Epoch 10: val_loss did not improve from 2.30068
79/79 ━━━━━━━━━━━━━━━━━━━━ 13s 127ms/step - categoric

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - categorical_accuracy: 0.0920 - loss: 2.3057


,id,label,total_layers,summary,conv_kernels,hidden_dense_units,train_minutes,test_loss,test_accuracy,best_model_h5,final_model_h5,layer_names
0,compact-7,Compact 9-layer,9,"2 conv, 1 relu, 1 sigmoid, 0 max-pool, 1 avg-p...",6,"[16, 8]",5.84,1.581247,0.472,/content/tiny-vgg/trained_models/compact-7/bes...,/content/tiny-vgg/trained_models/compact-7/fin...,"[conv_1_1, avg_pool_1, sigmoid_2_1, conv_2_1, ..."
1,balanced-12,Balanced 14-layer,14,"4 conv, 2 relu, 2 sigmoid, 1 max-pool, 1 avg-p...",10,"[16, 8]",2.96,2.302649,0.116,/content/tiny-vgg/trained_models/balanced-12/b...,/content/tiny-vgg/trained_models/balanced-12/f...,"[conv_1_1, avg_pool_1, sigmoid_2_1, conv_2_1, ..."
2,deep-17,Deep 19-layer,19,"6 conv, 3 relu, 3 sigmoid, 1 max-pool, 2 avg-p...",14,"[16, 8]",3.39,2.305742,0.092,/content/tiny-vgg/trained_models/deep-17/best.h5,/content/tiny-vgg/trained_models/deep-17/final.h5,"[conv_1_1, avg_pool_1, sigmoid_2_1, conv_2_1, ..."


## Export All Models to TensorFlow.js

This cell exports every trained model into its own folder and writes a minimal `model-index.json` file that the website can use for a dropdown selector.

The manifest intentionally stores only:

- `id`
- `label`
- `modelPath`

The website derives layer counts, summaries, visible layer lists, and stage boundaries by reading each selected `model.json` at runtime.


In [ ]:
import tensorflowjs as tfjs

manifest = []

for run in training_runs:
    export_dir = TFJS_ROOT / run['id']
    if export_dir.exists():
        shutil.rmtree(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    tfjs.converters.save_keras_model(trained_models[run['id']], str(export_dir))

    # Task 5: keep the selector manifest minimal. The website derives
    # total layers, summary, layer order, and stage boundaries from model.json.
    manifest.append({
        'id': run['id'],
        'label': run['label'],
        'modelPath': f"assets/data/models/{run['id']}/model.json",
    })

manifest_path = TFJS_ROOT / 'model-index.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print('Export root:', TFJS_ROOT)
print('Manifest:', manifest_path)
print(json.dumps(manifest, indent=2))


failed to lookup keras version from the file,
    this is likely a weight only file
failed to lookup keras version from the file,
    this is likely a weight only file
failed to lookup keras version from the file,
    this is likely a weight only file
Export root: /content/tiny-vgg/frontend_models
Manifest: /content/tiny-vgg/frontend_models/model-index.json
[
  {
    "id": "compact-7",
    "label": "Compact 9-layer",
    "modelPath": "assets/data/models/compact-7/model.json"
  },
  {
    "id": "balanced-12",
    "label": "Balanced 14-layer",
    "modelPath": "assets/data/models/balanced-12/model.json"
  },
  {
    "id": "deep-17",
    "label": "Deep 19-layer",
    "modelPath": "assets/data/models/deep-17/model.json"
  }
]


In [ ]:
# Optional: bundle the exported frontend assets for download or copying to Drive
# Task 3 at lines 2305-2312: bundle the exported frontend-ready models so the
# website can demonstrate all three trained architectures from one copied folder.
bundle_dir = WORKDIR / 'frontend_bundle'
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

models_target = bundle_dir / 'models'
shutil.copytree(TFJS_ROOT, models_target)

zip_path = WORKDIR / 'cnn_explainer_multi_model_exports.zip'
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', bundle_dir)
print('Created bundle:', zip_path)


Created bundle: /content/tiny-vgg/cnn_explainer_multi_model_exports.zip


## Frontend Handoff

After training finishes, copy the exported files into the website repo using this structure:

```text
public/assets/data/
  model-index.json
  models/
    compact-7/
      model.json
      group1-shard1of1.bin
    balanced-12/
      model.json
      group1-shard1of1.bin
    deep-17/
      model.json
      group1-shard1of1.bin
```

Then the website can load the manifest, let the user choose a model, and redraw the layers based on the selected architecture.

## About `.bin`-only architecture reading

A TensorFlow.js `.bin` file is only a raw binary weight shard. It does **not** store layer order, layer names, tensor names, tensor shapes, activations, pooling sizes, or model topology. Therefore, the website cannot reconstruct the architecture from `.bin` alone.

The minimal correct input for architecture-driven rendering is:

- model folder path,
- `model.json`,
- referenced `.bin` shard(s).

The frontend should derive architecture information from `model.json`, and use `.bin` only for trained weight values.


In [ ]:
# Copy the bundled zip to Google Drive
from pathlib import Path
import shutil

zip_path = WORKDIR / 'cnn_explainer_multi_model_exports.zip'
drive_zip_path = DRIVE_PROJECT_DIR / 'cnn_explainer_multi_model_exports.zip'

if not zip_path.exists():
    raise FileNotFoundError(f'Missing bundle zip: {zip_path}')

shutil.copy2(zip_path, drive_zip_path)
print('Copied zip to Drive:', drive_zip_path)


Copied zip to Drive: /content/drive/MyDrive/CNN/cnn_explainer_multi_model_exports.zip
